# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnanyaGujral/flyrank-ml-internship-start/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

Runs offline on the bundled slice. Three trailing signals, each with a mini-test and a verdict: **CONFIRMED / OPPOSITE / MIXED / FALSE**.

## 1. Distributions

Look before deciding. Note the heavy tails and the *systematic* missingness.

In [1]:
# ---- shared offline setup: bundled slice, leak-free pool + features, transparent rule ----
import json
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
MIN_PREV_IMPRESSIONS = 100     # "already visible" floor on the TRAILING window
DROP_FACTOR = 0.8              # declined = next-30 impressions < 80% of prior-30

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
DATA = root / "data" / "raw" / "content_refresh_anonymized.csv"
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(DATA)


def build_pool(frame):
    """Already-visible pages with a usable trailing window, plus leak-free features.

    Every feature is measured in the PRIOR 30-day window (days 31-60) or is static
    metadata. Nothing is taken from the 90-day totals or the outcome window, because
    those overlap the label's comparison window.
    """
    pool = frame[frame["impressions_prev_30d"] >= MIN_PREV_IMPRESSIONS].copy()
    pool["declined_next_30d"] = (
        pool["impressions_last_30d"] < DROP_FACTOR * pool["impressions_prev_30d"]).astype(int)
    pool["log_impr_prev"] = np.log1p(pool["impressions_prev_30d"])
    pool["log_clk_prev"] = np.log1p(pool["clicks_prev_30d"])
    pool["log_sess_prev"] = np.log1p(pool["sessions_prev_30d"])
    pool["ctr_prev"] = 100 * pool["clicks_prev_30d"] / pool["impressions_prev_30d"]
    pool["clk_per_sess_prev"] = pool["clicks_prev_30d"] / pool["sessions_prev_30d"].replace(0, np.nan)
    pool["has_keyword"] = pool["search_volume"].notna().astype(int)
    pool["has_clicks_prev"] = (pool["clicks_prev_30d"] > 0).astype(int)
    pool["has_sessions_prev"] = (pool["sessions_prev_30d"] > 0).astype(int)
    g = pool.groupby("client_id")
    pool["ctr_prev_z"] = ((pool["ctr_prev"] - g["ctr_prev"].transform("mean"))
                          / g["ctr_prev"].transform("std").replace(0, np.nan))
    pool["impr_prev_share"] = pool["impressions_prev_30d"] / g["impressions_prev_30d"].transform("sum")
    pool["clk_prev_share"] = pool["clicks_prev_30d"] / g["clicks_prev_30d"].transform("sum").replace(0, np.nan)
    return pool


NUM_FEATURES = ["log_impr_prev", "log_clk_prev", "log_sess_prev", "ctr_prev", "clk_per_sess_prev",
                "ctr_prev_z", "impr_prev_share", "clk_prev_share",
                "search_volume", "competition", "cpc", "word_count", "char_count",
                "content_age_days", "days_since_last_update",
                "has_keyword", "has_clicks_prev", "has_sessions_prev"]
CAT_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
                "freshness_tier", "word_count_tier"]


def feature_matrix(pool):
    Xn = pool[NUM_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([Xn, pd.get_dummies(pool[CAT_FEATURES].astype(str), dtype=float)], axis=1)


def rulescore(pool):
    """Transparent rule: low trailing CTR, weighted by trailing exposure."""
    med = float(pool["ctr_prev"].median())
    return (pool["ctr_prev"] <= 0.5 * med).to_numpy() * np.log1p(pool["impressions_prev_30d"].to_numpy())


pool = build_pool(raw)
X = feature_matrix(pool)
y = pool["declined_next_30d"].to_numpy()
groups = pool["client_id"].to_numpy()
rule = rulescore(pool)
print(f"bundled slice: {len(raw):,} rows -> pool {len(pool):,} pages | "
      f"{pool['client_id'].nunique()} pseudo-clients")
print(f"decline base rate in pool: {y.mean():.3f} | feature matrix: {X.shape}")
print("label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)")


def pk(table_scores, y_true, ks=(10, 20, 50, 100)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(table_scores))
    return {str(k): round(float(y_true[order[:k]].mean()), 3) for k in ks}

bundled slice: 30,000 rows -> pool 18,010 pages | 30 pseudo-clients
decline base rate in pool: 0.616 | feature matrix: (18010, 43)
label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)


In [2]:
cols = ["impressions_prev_30d", "clicks_prev_30d", "ctr_prev", "days_since_last_update",
        "word_count", "search_volume"]
desc = pool[cols].describe().T[["count", "mean", "50%", "max"]].round(3)
desc["missing"] = len(pool) - desc["count"]
print(f"pool rows: {len(pool):,}  |  decline base rate: {y.mean():.3f}")
print(desc.to_string())

pool rows: 18,010  |  decline base rate: 0.616
                          count      mean       50%         max  missing
impressions_prev_30d    18010.0  2955.421   797.000  218786.000      0.0
clicks_prev_30d         18010.0     9.010     1.000    1627.000      0.0
ctr_prev                18010.0     0.248     0.114       6.776      0.0
days_since_last_update  18010.0    52.446    22.000     313.000      0.0
word_count              12785.0  3448.944  2962.000    9546.000   5225.0
search_volume           17667.0   168.169    10.000   74000.000    343.0


**Read:** prior-window impressions and clicks are heavy-tailed; `ctr_prev` is small (percent units);
`word_count` and `search_volume` are missing for a large, *content-type-linked* share of rows — so any
threshold on them must state how it treats missing.

## 2. Signal test #1 / #2 / #3 (verdict each)

Each test: bucket the pool, print **n** next to the decline rate, and write a verdict. A signal with no
lift is reported as no lift.

In [3]:
# ---- Signal test #1: VOLUME. Does higher prior demand predict decline? ----
buckets = pd.qcut(pool["impressions_prev_30d"], 3, labels=["low", "mid", "high"])
sig1 = pool.groupby(buckets, observed=True).agg(n=("content_id", "count"),
                                                decline_rate=("declined_next_30d", "mean")).round(3)
print("SIGNAL CHECK 1 - prior-window volume")
print(sig1.to_string())
print()
print("VERDICT: MIXED - volume is a gate (it defines the pool) but not a monotone risk driver:")
print("the mid bucket is highest, so 'bigger = riskier' is FALSE on this slice. Use volume for")
print("exposure/impact, not as the decline score.")

SIGNAL CHECK 1 - prior-window volume
                         n  decline_rate
impressions_prev_30d                    
low                   6017         0.623
mid                   5989         0.652
high                  6004         0.573

VERDICT: MIXED - volume is a gate (it defines the pool) but not a monotone risk driver:
the mid bucket is highest, so 'bigger = riskier' is FALSE on this slice. Use volume for
exposure/impact, not as the decline score.


In [4]:
# ---- Signal test #2: CTR at a given exposure. Does low prior CTR predict decline? ----
mid = pool[pool["impressions_prev_30d"] >= 500].copy()
mid["ctr_band"] = pd.qcut(mid["ctr_prev"].rank(method="first"), 3, labels=["bottom", "mid", "top"])
sig2 = mid.groupby("ctr_band", observed=True).agg(n=("content_id", "count"),
                                                  decline_rate=("declined_next_30d", "mean")).round(3)
print("SIGNAL CHECK 2 - prior-window CTR (pages with >= 500 prior impressions)")
print(sig2.to_string())
print()
print("VERDICT: CONFIRMED - the bottom-CTR third declines noticeably more often than the top third.")
print("This is the one trailing edge a CTR-for-position-style rule can lean on.")

SIGNAL CHECK 2 - prior-window CTR (pages with >= 500 prior impressions)
             n  decline_rate
ctr_band                    
bottom    3706         0.722
mid       3706         0.603
top       3707         0.499

VERDICT: CONFIRMED - the bottom-CTR third declines noticeably more often than the top third.
This is the one trailing edge a CTR-for-position-style rule can lean on.


In [5]:
# ---- Signal test #3: FRESHNESS. Does 'not updated lately' predict decline? ----
fresh = pd.cut(pool["days_since_last_update"], [-1, 30, 90, 180, np.inf],
               labels=["<30d", "30-90d", "90-180d", "180d+"])
sig3 = pool.groupby(fresh, observed=True).agg(n=("content_id", "count"),
                                              decline_rate=("declined_next_30d", "mean"),
                                              median_prev_impr=("impressions_prev_30d", "median")).round(3)
print("SIGNAL CHECK 3 - staleness")
print(sig3.to_string())
print()
print("VERDICT: WEAK / NON-MONOTONE - the ordering is not clean and the oldest bucket is thin.")
print("Freshness stays a review cue, not a standalone decline flag.")

SIGNAL CHECK 3 - staleness
                            n  decline_rate  median_prev_impr
days_since_last_update                                       
<30d                    10900         0.608             725.0
30-90d                     85         0.588             336.0
90-180d                  7004         0.628             933.0
180d+                      21         0.952             527.0

VERDICT: WEAK / NON-MONOTONE - the ordering is not clean and the oldest bucket is thin.
Freshness stays a review cue, not a standalone decline flag.


## 3. The flag-linked test

Take the one flag the audit actually supports — "low trailing CTR on a visible page" — and test it
directly: is the decline rate **inside** the flag clearly above the rate **outside** it?

In [6]:
med = float(pool["ctr_prev"].median())
flags = {
    "low_ctr_visible": (pool["impressions_prev_30d"] >= 500) & (pool["ctr_prev"] <= 0.5 * med),
    "no_clicks_visible": (pool["impressions_prev_30d"] >= 500) & (pool["clicks_prev_30d"] == 0),
    "thin_visible": (pool["impressions_prev_30d"] >= 500) & (pool["word_count"].between(0.001, 1200)),
    "stale_visible": (pool["impressions_prev_30d"] >= 500) & (pool["days_since_last_update"] >= 180),
}
rows = []
for name, mask in flags.items():
    rows.append({"flag": name, "n": int(mask.sum()),
                 "decline_if_flagged": round(float(pool.loc[mask, "declined_next_30d"].mean()), 3),
                 "decline_if_not": round(float(pool.loc[~mask, "declined_next_30d"].mean()), 3),
                 "lift": round(float(pool.loc[mask, "declined_next_30d"].mean()
                                     - pool.loc[~mask, "declined_next_30d"].mean()), 3)})
flag_tbl = pd.DataFrame(rows)
print(f"base rate: {y.mean():.3f}")
print(flag_tbl.to_string(index=False))
print()
print("VERDICT: low_ctr_visible clears the bar (a real lift on a large slice). no_clicks_visible is")
print("weaker than expected once exposure is controlled. stale_visible is thin. thin_visible shows no lift.")

base rate: 0.616
             flag    n  decline_if_flagged  decline_if_not  lift
  low_ctr_visible 2772               0.737           0.594 0.144
no_clicks_visible 2043               0.736           0.600 0.136
     thin_visible   25               0.680           0.616 0.064
    stale_visible   11               1.000           0.616 0.384

VERDICT: low_ctr_visible clears the bar (a real lift on a large slice). no_clicks_visible is
weaker than expected once exposure is controlled. stale_visible is thin. thin_visible shows no lift.


## 4. What this means in practice

- **Keep:** trailing CTR below the exposure-adjusted midpoint (`low_ctr_visible`) — the one confirmed edge.
- **Gate, don't score:** prior-window volume sets exposure/impact; it is not a risk signal.
- **Demote:** staleness and thin content — useful context, no standalone lift here.
- **Honest word:** every verdict above is an *observed, in-snapshot* association with a defined label.
  The audit bounds what a rule may lean on; it does not license a causal claim about refresh.

In [7]:
metrics = {
    "task": "ml-06",
    "data": "data/raw/content_refresh_anonymized.csv",
    "pool_rows": int(len(pool)),
    "label": "declined_next_30d (prior-30 in, later-30 out)",
    "base_rate": round(float(y.mean()), 3),
    "signal_1_volume": {"verdict": "MIXED (gate, not driver)",
                        "decline_by_bucket": sig1["decline_rate"].to_dict()},
    "signal_2_ctr": {"verdict": "CONFIRMED",
                     "decline_by_band": sig2["decline_rate"].to_dict()},
    "signal_3_freshness": {"verdict": "WEAK / non-monotone",
                           "decline_by_band": sig3["decline_rate"].to_dict()},
    "flag_lift": flag_tbl.to_dict(orient="records"),
}
(OUT / "w04_signal_audit_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("receipt written: work/outputs/w04_signal_audit_metrics.json")

receipt written: work/outputs/w04_signal_audit_metrics.json


### Self-check

- [x] Distributions shown before any verdict
- [x] Three signals, each tested with n and a verdict
- [x] Flag-linked test with lift, not just a correlation story
- [x] Careful words throughout (observed / association / decision-support)
- [x] Receipt written to `work/outputs/w04_signal_audit_metrics.json`
- [x] Committed under `work/notebooks/`